In [ ]:
import sys
from pathlib import Path

if Path('/home/rocm-user/jupyter/nexus').exists():
    RUNNING_ON_SERVER = True
    ROOT = Path('/home/rocm-user/jupyter/nexus')
else:
    RUNNING_ON_SERVER = False
    ROOT = Path(r'C:/PersonalDrive/Programming/AiStudio/nexus-trader')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print({'running_on_server': RUNNING_ON_SERVER, 'root': str(ROOT)})


In [ ]:
from config.project_config import FEATURE_DIM_TOTAL, FEATURE_DIM_PRICE, FEATURE_DIM_NEWS, FEATURE_DIM_CROWD, FUSED_FEATURE_MATRIX_PATH, FUSED_TENSOR_PATH, TARGETS_PATH, FUSION_REPORT_PATH
print({
    'feature_dims': {'price': FEATURE_DIM_PRICE, 'news': FEATURE_DIM_NEWS, 'crowd': FEATURE_DIM_CROWD, 'total': FEATURE_DIM_TOTAL},
    'row_artifacts': [str(FUSED_FEATURE_MATRIX_PATH), str(TARGETS_PATH)],
    'optional_sequence_tensor': str(FUSED_TENSOR_PATH),
    'fusion_report': str(FUSION_REPORT_PATH),
    'note': 'Production training uses row-wise memmap artifacts plus lazy sequence windows to avoid giant fused tensors by default.',
})


In [ ]:
import subprocess

completed = subprocess.run(['python', 'scripts/build_fused_artifacts.py'], cwd=ROOT, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError('Fusion pipeline failed')


In [ ]:
import json
import numpy as np

report = json.loads(Path(FUSION_REPORT_PATH).read_text(encoding='utf-8'))
fused = np.load(FUSED_FEATURE_MATRIX_PATH, mmap_mode='r')
targets = np.load(TARGETS_PATH, mmap_mode='r')
print(report)
print({'fused_shape': tuple(fused.shape), 'target_shape': tuple(targets.shape)})
